# Journal des Expérimentations — C-MAPSS FD004

Ce document retrace les différentes itérations de modélisation pour l'estimation de la RUL sur le sous-ensemble FD004.

---

## Synthèse des itérations

| Expérience | Modèle | Prétraitement clé | CV RMSE (cycles) | Test RMSE (cycles) | Score NASA (Test) | Statut |
|---|---|---|---|---|---|---|
| **EXP-01** | Ridge Baseline | Standardisation globale brute | 39.2 | 34.8 | ~14 200 | Référence |
| **EXP-02** | Random Forest | K-Means (6 régimes) + Z-score local | 31.4 | 29.2 | ~7 850 | Amélioration |
| **EXP-03** | LightGBM | Normalisation régimes + Rolling 15 + Clip 125 | 25.3 | 24.1 | ~3 420 | **Champion** |
| **EXP-04** | XGBoost | Même pipeline + hist tree method | 25.6 | 24.4 | ~3 610 | Challenger |

---

## Détail des expérimentations

### EXP-01 : Baseline linéaire (Ridge)
* **Hypothèse :** Une régression linéaire avec régularisation L2 permet d'établir le seuil de performance minimal sans ingénierie complexe.
* **Données :** Capteurs bruts sans normalisation par régime de vol.
* **Constat :** Échec relatif dû à l'incapacité du modèle à distinguer les changements de conditions de vol de la dégradation mécanique.

### EXP-02 : Ensembling non-linéaire (Random Forest)
* **Hypothèse :** Les arbres de décision capturent mieux les interactions non-linéaires entre capteurs.
* **Ajouts :**
  - Clustering $K$-Means ($k=6$) sur `setting_1`, `setting_2`, `setting_3`.
  - Normalisation locale par régime ($z$-score conditionnel).
* **Constat :** Baisse notable du RMSE. Le modèle plafonne néanmoins face au volume de données (~61k lignes) et aux bruits résiduels.

### EXP-03 : Gradient Boosting régularisé (LightGBM) — Modèle retenu
* **Hypothèse :** La croissance par feuilles (*leaf-wise*) combinée à une régularisation L1/L2 (`reg_alpha=1.0`, `reg_lambda=5.0`) neutralise le bruit des capteurs secondaires.
* **Ajouts :**
  - Fenêtres glissantes temporelles (`rolling_mean_15`, `rolling_std_15`) par moteur.
  - Plafonnement de la cible à 125 cycles (*Piecewise linear target*).
  - Validation croisée sans fuite `GroupKFold` (5 splits sur `unit_id`).
  - Prédiction par moyenne d'ensemble des 5 modèles.
* **Résultat :** Meilleur compromis : RMSE de **24,1** cycles et division par 4 du score de pénalité NASA.

